# langchain-bigquery-graph: Basic Usage

This notebook demonstrates how to build a Graph RAG application with **BigQuery Graphs** using `langchain-bigquery-graph`.

You will learn how to:
1. Create a `BigQueryGraphStore` and populate it with graph documents
2. Use `BigQueryGraphVectorContextRetriever` for vector-based graph search
3. Use `BigQueryGraphTextToGQLRetriever` for natural language to GQL translation

### Prerequisites
- A Google Cloud project with BigQuery API enabled
- A BigQuery dataset created beforehand
- Authentication configured (`gcloud auth application-default login`)
- `GOOGLE_GENAI_USE_VERTEXAI=true` environment variable set

## Phase 1: Environment Setup

### Install Dependencies
Install `langchain-bigquery-graph` and required packages.

In [1]:
!pip install "langchain-bigquery-graph[examples] @ git+https://github.com/ksmin23/langchain-bigquery-graph.git" --quiet

### Configure Project Settings
Set your Google Cloud project ID, BigQuery dataset, and model names.

In [2]:
import os

PROJECT_ID = "your-gcp-project-id (e.g., dev***)"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
DATASET_ID = "your-bigquery-dataset (e.g., langchain_bg_graph)"  # @param {type:"string"}
GRAPH_NAME = "your-bigquery-graph-name (e.g., test_kg)"  # @param {type:"string"}
EMBEDDING_MODEL = "gemini-embedding-001"  # @param {type:"string"}
LLM_MODEL = "gemini-2.5-flash"  # @param {type:"string"}

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"

### Create BigQuery Dataset
The dataset must exist before using `BigQueryGraphStore`. Tables and property graphs are created automatically.

In [3]:
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{DATASET_ID}

## Phase 2: Create Graph Store & Ingest Data

### Initialize BigQueryGraphStore
Connect to your BigQuery dataset and create a graph store instance.

In [4]:
from langchain_bigquery_graph import BigQueryGraphStore

store = BigQueryGraphStore(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    graph_name=GRAPH_NAME,
    location=LOCATION,
)

print(f"Graph store created: project={PROJECT_ID}, dataset={DATASET_ID}, graph={GRAPH_NAME}")

Graph store created: project='dev***', dataset=langchain_bg_graph, graph=test_kg


### Generate Embeddings
Compute vector embeddings for each node using the Gemini embedding model. These embeddings enable vector similarity search on graph nodes.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

texts = [
    "Alice is a 30 year old person",
    "Bob is a 25 year old person",
    "Acme Corp is a company",
]
vectors = embeddings.embed_documents(texts)

print(f"Generated {len(vectors)} embeddings (dim={len(vectors[0])}) using {EMBEDDING_MODEL}")

Generated 3 embeddings (dim=3072) using gemini-embedding-001


### Build Graph Documents
Define nodes (`Person`, `Company`) and relationships (`WORKS_AT`, `KNOWS`) as `GraphDocument` objects.

```
alice (Person) --[WORKS_AT]--> acme (Company)
alice (Person) --[KNOWS]-----> bob  (Person)
```

In [6]:
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

alice = Node(id="alice", type="Person", properties={"name": "Alice", "age": 30, "embedding": vectors[0]})
bob = Node(id="bob", type="Person", properties={"name": "Bob", "age": 25, "embedding": vectors[1]})
acme = Node(id="acme", type="Company", properties={"name": "Acme Corp", "embedding": vectors[2]})

works_at = Relationship(source=alice, target=acme, type="WORKS_AT")
knows = Relationship(source=alice, target=bob, type="KNOWS")

doc = GraphDocument(
    nodes=[alice, bob, acme],
    relationships=[works_at, knows],
    source=Document(page_content="Alice works at Acme Corp and knows Bob."),
)

print(f"Nodes:         {', '.join(n.id + ' (' + n.type + ')' for n in doc.nodes)}")
print(f"Relationships: {', '.join(r.source.id + ' -[' + r.type + ']-> ' + r.target.id for r in doc.relationships)}")

Nodes:         alice (Person), bob (Person), acme (Company)
Relationships: alice -[WORKS_AT]-> acme, alice -[KNOWS]-> bob


### Ingest Graph Documents
This call creates BigQuery tables, defines the property graph DDL, and inserts all node/edge data.

- Node tables: `{GRAPH_NAME}_Person`, `{GRAPH_NAME}_Company`
- Edge tables: `{GRAPH_NAME}_Person_WORKS_AT_Company`, `{GRAPH_NAME}_Person_KNOWS_Person`
- Property graph: `{DATASET_ID}.{GRAPH_NAME}`

In [7]:
store.add_graph_documents([doc])

print("Graph documents ingested successfully.")
print(f"\nSchema:\n{store.get_schema}")

Graph documents ingested successfully.

Schema:
{
  "Name of graph": "langchain_bg_graph.test_kg",
  "Node properties per node label": {
    "Company": [
      {
        "name": "embedding",
        "type": "ARRAY<FLOAT64>"
      },
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "name",
        "type": "STRING"
      }
    ],
    "Person": [
      {
        "name": "age",
        "type": "INT64"
      },
      {
        "name": "embedding",
        "type": "ARRAY<FLOAT64>"
      },
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "name",
        "type": "STRING"
      }
    ]
  },
  "Edge properties per edge label": {
    "KNOWS": [
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "target_id",
        "type": "STRING"
      }
    ],
    "WORKS_AT": [
      {
        "name": "id",
        "type": "STRING"
      },
      {
        "name": "target_id",
        "type": "ST

### Query with GQL
Run a GQL query to retrieve all `Person` nodes from the property graph.

In [8]:
gql = f"GRAPH `{DATASET_ID}`.`{GRAPH_NAME}` MATCH (p:Person) RETURN p.name AS name, p.age AS age"
print(f"GQL: {gql}\n")

results = store.query(gql)
for row in results:
    print(f"  {row}")

GQL: GRAPH `langchain_bg_graph`.`test_kg` MATCH (p:Person) RETURN p.name AS name, p.age AS age

  {'name': 'Bob', 'age': 25}
  {'name': 'Alice', 'age': 30}


### Query Relationships
Traverse the graph to find connections between nodes.

In [9]:
gql = (
    f"GRAPH `{DATASET_ID}`.`{GRAPH_NAME}` "
    "MATCH (a:Person)-[r]->(b) "
    "RETURN a.name AS source, LABELS(r) AS relationship, b.name AS target"
)
print(f"GQL: {gql}\n")

results = store.query(gql)
for row in results:
    print(f"  {row}")

GQL: GRAPH `langchain_bg_graph`.`test_kg` MATCH (a:Person)-[r]->(b) RETURN a.name AS source, LABELS(r) AS relationship, b.name AS target

  {'source': 'Alice', 'relationship': ['WORKS_AT'], 'target': 'Acme Corp'}
  {'source': 'Alice', 'relationship': ['KNOWS'], 'target': 'Bob'}


## Phase 3: Vector Context Retriever

Search graph nodes by vector similarity and expand results by traversing the graph neighborhood.

**How it works:**
1. Vector search runs as SQL on the base table (BigQuery property graphs don't support ARRAY properties)
2. Matching node IDs are used to traverse the graph via GQL
3. Results include the matched nodes and their neighbors up to `expand_by_hops` distance

In [10]:
from langchain_bigquery_graph import BigQueryGraphVectorContextRetriever

vector_retriever = BigQueryGraphVectorContextRetriever.from_params(
    embedding_service=embeddings,
    graph_store=store,
    label_expr="Person",
    embeddings_column="embedding",
    expand_by_hops=1,
    top_k=5,
    k=10,
)

print("Config: label_expr=Person, expand_by_hops=1, top_k=5, distance=COSINE")

Config: label_expr=Person, expand_by_hops=1, top_k=5, distance=COSINE


In [11]:
question = "Who works at Acme?"
print(f"Question: \"{question}\"\n")

docs = vector_retriever.invoke(question)

print(f"Retrieved {len(docs)} document(s):")
for i, doc in enumerate(docs, 1):
    print(f"  [{i}] {doc.page_content}")

Question: "Who works at Acme?"

Retrieved 5 document(s):
  [1] [{"kind": "node", "labels": ["Person"], "properties": {"age": 30, "id": "alice", "name": "Alice"}}, {"kind": "edge", "labels": ["WORKS_AT"], "properties": {"id": "alice", "target_id": "acme"}}, {"kind": "node", "labels": ["Company"], "properties": {"id": "acme", "name": "Acme Corp"}}]
  [2] [{"kind": "node", "labels": ["Person"], "properties": {"age": 25, "id": "bob", "name": "Bob"}}]
  [3] [{"kind": "node", "labels": ["Person"], "properties": {"age": 30, "id": "alice", "name": "Alice"}}]
  [4] [{"kind": "node", "labels": ["Person"], "properties": {"age": 30, "id": "alice", "name": "Alice"}}, {"kind": "edge", "labels": ["KNOWS"], "properties": {"id": "alice", "target_id": "bob"}}, {"kind": "node", "labels": ["Person"], "properties": {"age": 25, "id": "bob", "name": "Bob"}}]
  [5] [{"kind": "node", "labels": ["Person"], "properties": {"age": 25, "id": "bob", "name": "Bob"}}, {"kind": "edge", "labels": ["KNOWS"], "properties"

## Phase 4: Text-to-GQL Retriever

Translate natural language questions into GQL queries using an LLM, execute them against the property graph, and return the results.

**How it works:**
1. The graph schema is passed to the LLM as context
2. The LLM generates a GQL query from the natural language question
3. The generated GQL is executed against BigQuery
4. Few-shot examples can be provided to improve query generation

### Initialize Retriever

In [ ]:
from langchain_google_vertexai import ChatVertexAI
from langchain_bigquery_graph import BigQueryGraphTextToGQLRetriever

llm = ChatVertexAI(model=LLM_MODEL)
print(f"LLM: {LLM_MODEL}")

text2gql_retriever = BigQueryGraphTextToGQLRetriever.from_params(
    llm=llm,
    embedding_service=embeddings,
    graph_store=store,
    k=10,
)

print("Text-to-GQL retriever initialized.")

### Add Few-Shot Example
Provide a sample question-GQL pair to guide the LLM's query generation. The retriever uses semantic similarity to select the most relevant examples at query time.

In [13]:
example_gql = (
    f"GRAPH `{DATASET_ID}`.`{GRAPH_NAME}` "
    "MATCH (p:Person)-[:WORKS_AT]->(c:Company {name: 'Acme Corp'}) "
    "RETURN p.name AS name"
)

text2gql_retriever.add_example(
    question="Who works at Acme?",
    gql=example_gql,
)

print(f"Example Q:   \"Who works at Acme?\"")
print(f"Example GQL: {example_gql}")

Example Q:   "Who works at Acme?"
Example GQL: GRAPH `langchain_bg_graph`.`test_kg` MATCH (p:Person)-[:WORKS_AT]->(c:Company {name: 'Acme Corp'}) RETURN p.name AS name


### Invoke Text-to-GQL
Ask a new question. The LLM generates GQL using the graph schema and few-shot examples, then executes it.

In [14]:
question = "Find all people who work at Acme Corp"
print(f"Question: \"{question}\"\n")

docs = text2gql_retriever.invoke(question)

print(f"Retrieved {len(docs)} document(s):")
for i, doc in enumerate(docs, 1):
    print(f"  [{i}] {doc.page_content}")

Question: "Find all people who work at Acme Corp"

Retrieved 1 document(s):
  [1] {"name": "Alice"}


## Cleanup (Optional)
Remove the property graph and all associated tables from BigQuery.

In [ ]:
# Uncomment to clean up
# store.cleanup()
# print("Property graph and all associated tables removed.")